# 회로 분석: 계산 서브그래프 찾기 - 회로(Circuit)란 무엇인가

- Tutorial ID: `tut-13-1`
- Tutorial: 회로 분석: 계산 서브그래프 찾기
- Section ID: `s13-1-1`
- Section: 회로(Circuit)란 무엇인가


In [ ]:
# ============================================================
# 코드 읽는 법 — 회로(Circuit)란 무엇인가
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#   3) 미래 토큰을 -inf로 막은 뒤 softmax 확률이 0이 되는지 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

print("=" * 55)
print("회로 분석: 계산 서브그래프 찾기")
print("=" * 55)

def softmax(x, axis=-1):
        x = x - np.max(x, axis=axis, keepdims=True)
    return np.exp(x) / np.sum(np.exp(x), axis=axis, keepdims=True)

vocab_size = 6
d_model = 8
d_head = 4
vocab = ['A', 'B', 'C', 'D', 'E', 'F']

np.random.seed(42)

W_E = np.random.randn(vocab_size, d_model) * 0.4
W_U = np.random.randn(d_model, vocab_size) * 0.3

# 2개의 헤드 (레이어 0, 1)
W_Q = [np.random.randn(d_model, d_head) * 0.3 for _ in range(2)]
W_K = [np.random.randn(d_model, d_head) * 0.3 for _ in range(2)]
W_V = [np.random.randn(d_model, d_head) * 0.3 for _ in range(2)]
W_O = [np.random.randn(d_head, d_model) * 0.3 for _ in range(2)]

In [ ]:
# 직접 로짓 기여 (Direct Logit Attribution, DLA)

In [ ]:
print("\n1. 직접 로짓 기여 분석 (DLA)")
print("-" * 40)

# 시퀀스 A B A → 다음 토큰 예측
seq = [0, 1, 0]  # A, B, A
X = W_E[seq]
mask = np.triu(np.full((3, 3), -1e9), k=1)

# 각 레이어의 전방 패스
def attn_head_output(X, Wq, Wk, Wv, Wo, mask):
        Q = X @ Wq
        K = X @ Wk
        V = X @ Wv
        A = softmax(Q @ K.T / np.sqrt(d_head) + mask)
    return (A @ V) @ Wo, A

# 레이어 0
head0_out, A0 = attn_head_output(X, W_Q[0], W_K[0], W_V[0], W_O[0], mask)
X1 = X + head0_out  # 잔차

# 레이어 1
head1_out, A1 = attn_head_output(X1, W_Q[1], W_K[1], W_V[1], W_O[1], mask)
X2 = X1 + head1_out

# 최종 로짓
logits = X2[-1] @ W_U  # 마지막 위치

# DLA: 각 구성 요소의 기여
# logits = (X[-1] + head0_out[-1] + head1_out[-1]) @ W_U
direct_contrib = X[-1] @ W_U          # 원래 임베딩 기여
head0_contrib = head0_out[-1] @ W_U   # 레이어0 헤드 기여
head1_contrib = head1_out[-1] @ W_U   # 레이어1 헤드 기여

print(f"입력: A B A → 다음 토큰 예측")
print(f"\n각 구성 요소의 로짓 기여:")
print(f"  직접(임베딩):  {np.round(direct_contrib, 2)}")
print(f"  레이어0 헤드:  {np.round(head0_contrib, 2)}")
print(f"  레이어1 헤드:  {np.round(head1_contrib, 2)}")
print(f"  합계:          {np.round(logits, 2)}")
print(f"\n  검증: {np.allclose(direct_contrib + head0_contrib + head1_contrib, logits)}")

# 각 헤드가 어떤 토큰의 확률을 높이는가
print(f"\n레이어0 헤드가 가장 강화하는 토큰: {vocab[np.argmax(head0_contrib)]}")
print(f"레이어1 헤드가 가장 강화하는 토큰: {vocab[np.argmax(head1_contrib)]}")
print(f"최종 예측: {vocab[np.argmax(logits)]}")

In [ ]:
# 활성화 패칭 (Activation Patching)

In [ ]:
print("\n2. 활성화 패칭 (인과적 추적)")
print("-" * 40)
print("""
활성화 패칭의 원리:
  1. "깨끗한" 입력으로 실행 (정답)
  2. "오염된" 입력으로 실행 (다른 입력)
  3. 오염된 실행에서 특정 활성화를 깨끗한 것으로 교체
  4. 예측이 얼마나 변하는지 측정
  → 그 활성화가 얼마나 중요한지 알 수 있음!

예시:
  깨끗한 입력: "A B A" → 예측: B (인덕션)
  오염된 입력: "C D C" → 예측: D

  레이어0 헤드 출력을 교체하면:
    → 예측이 얼마나 변하는가?
  레이어1 헤드 출력을 교체하면:
    → 예측이 얼마나 변하는가?
  
  기여가 큰 구성 요소 = 중요한 회로 부품!
""")

In [ ]:
# 회로 최소화 (Circuit Minimization)

In [ ]:
print("3. 회로 최소화: 최소 필요 구성요소 찾기")
print("-" * 40)
print("""
회로 분석의 핵심:
  특정 기능을 위해 꼭 필요한 최소 구성 요소는?

인덕션 회로의 최소 구성요소:
  ✅ 이전 토큰 헤드 (레이어 0의 특정 헤드)
  ✅ 인덕션 헤드 (레이어 1의 특정 헤드)
  ❌ 나머지 헤드들은 제거해도 거의 영향 없음

판별 방법:
  - 각 헤드를 제거(ablation)하고 성능 변화 측정
  - "zero ablation": 해당 헤드 출력을 0으로 설정
  - "mean ablation": 평균 출력으로 대체
  - "random ablation": 랜덤 입력의 출력으로 대체

성능이 크게 떨어지는 헤드 = 회로의 필수 부품!
""")

# 간단한 ablation 시뮬레이션
print("어블레이션 시뮬레이션 (예시):")
base_logits = logits.copy()
base_pred = np.argmax(base_logits)
print(f"  기준 예측: {vocab[base_pred]} (로짓={base_logits[base_pred]:.3f})")

# 레이어0 제거
logits_no_h0 = (X[-1] + head1_out[-1]) @ W_U
pred_no_h0 = np.argmax(logits_no_h0)
change = logits_no_h0[base_pred] - base_logits[base_pred]
print(f"  레이어0 제거: {vocab[pred_no_h0]} (로짓 변화: {change:+.3f})")

# 레이어1 제거
logits_no_h1 = (X[-1] + head0_out[-1]) @ W_U
pred_no_h1 = np.argmax(logits_no_h1)
change = logits_no_h1[base_pred] - base_logits[base_pred]
print(f"  레이어1 제거: {vocab[pred_no_h1]} (로짓 변화: {change:+.3f})")